# 🌳 Day 01: Tree Fundamentals — Thinking Recursively

---

## Why Trees Are Different

Arrays and linked lists are linear — one element after another. Trees branch out. And that branching is why **recursion** becomes your primary tool.

Every tree problem boils down to: do something at the current node, then recurse on children. The magic is knowing **when** to process (before kids? after kids? between kids?).

```
         1            Binary Tree: each node has at most 2 children
        / \
       2   3          Terminology:
      / \   \         • Root: node 1 (no parent)
     4   5   6        • Leaf: nodes 4, 5, 6 (no children)
                      • Height: longest root-to-leaf path (here: 2)
                      • Depth of node 5: 2 (edges from root)
```

---

In [ ]:
# ==========================================
# TreeNode — you'll use this in EVERY problem
# ==========================================
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Helper: build a tree from a level-order list (LeetCode style)
from collections import deque

def build_tree(values):
    """Build tree from list like [1,2,3,None,5,None,6]."""
    if not values:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root

def print_tree(root, level=0, prefix="Root: "):
    """Pretty-print a tree."""
    if root is not None:
        print(" " * (level * 4) + prefix + str(root.val))
        if root.left or root.right:
            print_tree(root.left, level + 1, "L--- ")
            print_tree(root.right, level + 1, "R--- ")

# Build and visualize
tree = build_tree([1, 2, 3, 4, 5, None, 6])
print_tree(tree)

---

## 🎯 Problem 1: Maximum Depth of Binary Tree (LC #104)

The simplest tree recursion. Height = 1 + max(height of left subtree, height of right subtree).

**Base case:** Empty node has depth 0. This is important — the base case is what stops the recursion.

In [ ]:
def maxDepth(root):
    """
    Time: O(n), Space: O(h) where h = height
    """
    if not root:
        return 0
    return 1 + max(maxDepth(root.left), maxDepth(root.right))

tree = build_tree([3, 9, 20, None, None, 15, 7])
print(f"Max depth: {maxDepth(tree)}")  # 3

---

## 🎯 Problem 2: Invert Binary Tree (LC #226)

Swap left and right children at every node. The famous problem Homebrew's author couldn't solve.

```
     4              4
    / \    →       / \
   2   7          7   2
  / \ / \        / \ / \
 1  3 6  9      9  6 3  1
```

In [ ]:
def invertTree(root):
    """
    Swap children, then recurse. That's it.
    Time: O(n), Space: O(h)
    """
    if not root:
        return None
    root.left, root.right = root.right, root.left
    invertTree(root.left)
    invertTree(root.right)
    return root

tree = build_tree([4, 2, 7, 1, 3, 6, 9])
print("Before:")
print_tree(tree)
invertTree(tree)
print("\nAfter invert:")
print_tree(tree)

---

## 🎯 Problem 3: Same Tree (LC #100) & Symmetric Tree (LC #101)

**Same Tree:** Two trees are same if roots match AND left subtrees are same AND right subtrees are same.

**Symmetric Tree:** A tree is symmetric if its left subtree is a mirror of its right subtree. "Mirror" means: left.left matches right.right, and left.right matches right.left.

In [ ]:
def isSameTree(p, q):
    """Both None → True. One None → False. Vals match + recurse both sides."""
    if not p and not q:
        return True
    if not p or not q:
        return False
    return (p.val == q.val and 
            isSameTree(p.left, q.left) and 
            isSameTree(p.right, q.right))

def isSymmetric(root):
    """Mirror check: left.left ↔ right.right, left.right ↔ right.left"""
    def isMirror(left, right):
        if not left and not right:
            return True
        if not left or not right:
            return False
        return (left.val == right.val and 
                isMirror(left.left, right.right) and   # Outer
                isMirror(left.right, right.left))      # Inner
    
    return isMirror(root.left, root.right) if root else True

# Test Same Tree
t1 = build_tree([1, 2, 3])
t2 = build_tree([1, 2, 3])
print(f"Same tree: {isSameTree(t1, t2)}")  # True

# Test Symmetric
sym = build_tree([1, 2, 2, 3, 4, 4, 3])
print(f"Symmetric: {isSymmetric(sym)}")  # True

asym = build_tree([1, 2, 2, None, 3, None, 3])
print(f"Symmetric: {isSymmetric(asym)}")  # False

---

## 🎯 Problem 4: Balanced Binary Tree (LC #110)

A tree is balanced if for **every** node, the height difference between left and right subtrees is at most 1.

**Naive approach:** Check height at each node → O(n²). **Optimized:** Compute height bottom-up, return -1 if imbalanced.

In [ ]:
def isBalanced(root):
    """
    Bottom-up: return height if balanced, -1 if not.
    Time: O(n), Space: O(h)
    """
    def check(node):
        if not node:
            return 0
        
        left_h = check(node.left)
        if left_h == -1:
            return -1  # Left subtree already imbalanced
        
        right_h = check(node.right)
        if right_h == -1:
            return -1  # Right subtree already imbalanced
        
        if abs(left_h - right_h) > 1:
            return -1  # Current node imbalanced
        
        return 1 + max(left_h, right_h)
    
    return check(root) != -1

balanced = build_tree([3, 9, 20, None, None, 15, 7])
print(f"Balanced: {isBalanced(balanced)}")  # True

unbalanced = build_tree([1, 2, 2, 3, 3, None, None, 4, 4])
print(f"Balanced: {isBalanced(unbalanced)}")  # False

**The -1 trick** is a classic pattern: use the return value to carry both the answer AND an error signal. You'll see this in many tree problems.

---

## 🗺️ Tree Fundamentals Pattern Map

```
╔══════════════════════════════════════════════════════════════════════╗
║  TREE FUNDAMENTALS — The Recursion Template                         ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  TEMPLATE:                                                           ║
║    def solve(node):                                                  ║
║        if not node: return BASE_CASE                                 ║
║        left = solve(node.left)                                       ║
║        right = solve(node.right)                                     ║
║        return COMBINE(node.val, left, right)                         ║
║                                                                      ║
║  PATTERNS:                                                           ║
║  • Depth/Height → 1 + max(left, right)                             ║
║  • Same/Mirror  → match vals + recurse children                     ║
║  • Balanced     → bottom-up height, -1 = imbalanced                 ║
║  • Invert       → swap children then recurse                        ║
║                                                                      ║
║  SPACE: O(h) for recursion stack, h = log n balanced, n worst case  ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
```

---

## 📝 Practice Problems

| # | Problem | Difficulty | Key Idea |
|---|---------|------------|----------|
| 104 | Maximum Depth | Easy | 1 + max(left, right) |
| 226 | Invert Binary Tree | Easy | Swap + recurse |
| 100 | Same Tree | Easy | Match vals + both subtrees |
| 101 | Symmetric Tree | Easy | Mirror: outer + inner |
| 110 | Balanced Binary Tree | Easy | Bottom-up, -1 trick |
| 111 | Minimum Depth | Easy | Careful: leaf node only |
| 572 | Subtree of Another Tree | Easy | isSameTree at each node |

---

## ✅ Key Takeaways

```
┌──────────────────────────────────────────────────────────────────────┐
│  📌 Trees = Recursion. Base case is always: if not node: return    │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Most tree problems: solve left, solve right, combine          │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Bottom-up returns info. Top-down passes info as parameters.   │
├──────────────────────────────────────────────────────────────────────┤
│  📌 Space is O(h): O(log n) balanced, O(n) skewed                │
└──────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Tree Traversals

Preorder, Inorder, Postorder, Level Order — both recursive and iterative.